# Sequential pricing: what would a 12% price rise do to renewals?

## At a glance

A subscription retailer's pricing engine sets each subscriber's price once a month for three
months, and then the subscriber renews or lapses. Would raising each price by 12%, wherever the
raised price stays inside the range the engine uses, change month-3 renewals and revenue? Nobody
randomized the prices.

Compared with the engine's prices, the rise would cut the share who renew in month 3 by about
10 percentage points (estimate -0.10, 95% interval -0.115 to -0.090). It would lower mean month-3
revenue by about 0.31 per subscriber (interval -0.42 to -0.20). One regression of renewal on the
three prices understates the renewal effect by about a fifth, because it holds the later months
fixed.

The answer is causal only if the engine's log records every input that moved both a price and
the renewal, which the data cannot check. The data are simulated, so we know the true effects:
-0.100 on renewal and -0.31 on revenue.

## The question

Each subscriber starts a three-month term at time zero and commits to three monthly payments, so
no one can leave before the term ends. Each month the engine scores whether the subscriber is
engaged, then draws a price inside a band from $4 + 2e$ to $8 + 2e$, where $e$ is 1 for an engaged
subscriber and 0 otherwise. After the third price the subscriber renews or lapses, and month-3
revenue is the month-3 price for a renewer and 0 otherwise.

The pricing lead asks: among subscribers who start a three-month term at time zero, what share
would renew in month 3, and what would month-3 revenue be, if each monthly price had been 12%
higher, compared with the engine's own prices? The rise applies only where the raised price stays
inside that month's band, and the engine's price stands elsewhere. A second plan cuts each price
by 15% wherever the cut price stays inside the band.

The data hold no price outside the band, so the fit refuses a plain 12% rise: that price lies
outside the declared band. A rule that raised the price and capped it at the ceiling would send a
whole range of prices to one value. The method needs each piece of a rule to map different prices
to different prices (Assumption 4 of
[Díaz, Williams, Hoffman and Schenck 2023](https://doi.org/10.1080/01621459.2021.1955691)).

The graph shows what we assume causes what, including what else the engine reads.

```text
  usage        ──▶  every engagement score, every price, and the renewal

  in each month t = 1, 2, 3:
  engaged_t    ──▶  price_t     the band rises with engagement
  price_t-1    ──▶  price_t     the engine carries part of last month's price gap
  engaged_t-1  ──▶  price_t     it measures that gap from last month's band
  price_t-1    ──▶  engaged_t   a price above the band's middle lowers engagement
  engaged_t-1  ──▶  engaged_t   engagement persists
  engaged_t    ──▶  renewed
  price_t      ──▶  renewed     month 3's price matters most

  not drawn: an unrecorded cause U with U ──▶ price and U ──▶ renewed
```

Engagement makes this hard. It drives the next price and the renewal, so the analysis must adjust
for it, but earlier prices change it, so it also carries part of the effect we want. The missing U
is the key assumption: the engine logs every input it reads, so nothing unrecorded moves both a
price and the renewal.

## The data

Every fit sets its seeds, so a rerun prints the same numbers.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.5


Another version of `cleverly` can print slightly different numbers.

`make_sequential_pricing` draws 10,000 simulated subscribers and also returns the true effects,
which real data never give you. The seed was fixed before the page ran, not chosen for its
results.

In [2]:
from cleverly.datasets import make_sequential_pricing

# With your own data, load one row per subscriber here, such as frame = pd.read_csv(...),
# with each month's engagement and price as separate columns in time order.
# Real data have no `truth`, so drop every line that uses it.
frame, truth = make_sequential_pricing(10_000, seed=1000)
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head(6).round(3))
print()
print("missing values:", int(frame.isna().sum().sum()))
print("known values of the synthetic law:")
for plan in ("natural_prices", "price_up", "price_down"):
    renewal = truth[f"ey_regimen[{plan}]"]
    revenue = truth[f"ey_revenue[{plan}]"]
    print(f"  {plan:15s} mean renewal {renewal:.4f}  mean revenue {revenue:.4f}")
for plan in ("price_up", "price_down"):
    renewal = truth[f"ate_regimen[{plan} vs natural_prices]"]
    revenue = truth[f"ate_revenue[{plan} vs natural_prices]"]
    print(f"  {plan} against natural_prices: renewal {renewal:+.4f}  revenue {revenue:+.4f}")

rows and columns: (10000, 9)
['usage', 'engaged_1', 'price_1', 'engaged_2', 'price_2', 'engaged_3', 'price_3', 'renewed', 'revenue']
   usage  engaged_1  price_1  engaged_2  price_2  engaged_3  price_3  renewed  revenue
0  0.053        0.0    6.586        1.0    9.119        1.0    7.696      1.0    7.696
1  0.260        0.0    4.130        0.0    6.248        1.0    6.872      1.0    6.872
2 -0.072        1.0    8.736        0.0    6.877        1.0    7.356      1.0    7.356
3 -0.817        1.0    7.768        1.0    8.664        1.0    8.672      1.0    8.672
4  0.071        1.0    7.596        0.0    4.978        0.0    7.147      0.0    0.000
5 -0.860        0.0    6.818        0.0    4.364        1.0    6.299      0.0    0.000

missing values: 0
known values of the synthetic law:
  natural_prices  mean renewal 0.7301  mean revenue 5.3765
  price_up        mean renewal 0.6298  mean revenue 5.0643
  price_down      mean renewal 0.8450  mean revenue 5.5019
  price_up against natural_

Each row holds `usage`, a usage score, then each month's engagement score (1 engaged, 0 not) and
price, then `renewed` (1 or 0) and `revenue`, which is `renewed` times `price_3`. No value is
missing. `pricing_truth()` computes the true values exactly. Under the engine's own prices
(`natural_prices`) the renewal share is 0.730 and mean revenue 5.38. The 12% rise (`price_up`)
changes them by -0.100 and -0.31, and the 15% cut (`price_down`) by 0.115 and 0.125.

## Why a simple comparison misleads

Engaged subscribers get higher prices and also renew more often, so in raw data a high price can
look harmless.

In [3]:
gap_1 = frame["price_1"] - (6 + 2 * frame["engaged_1"])
print("share engaged in month 2, by the month-1 price against the middle of its band:")
print(
    frame.groupby(np.where(gap_1 > 0, "above the middle", "at or below"))["engaged_2"]
    .mean()
    .round(3)
    .to_string()
)
print()
print("mean price by that month's engagement (0 = not engaged, 1 = engaged):")
by_engagement = {
    f"month {t}": frame.groupby(f"engaged_{t}")[f"price_{t}"].mean() for t in (1, 2, 3)
}
print(pd.DataFrame(by_engagement).round(3).to_string())
print()
print("share renewed, by month-3 engagement:")
print(frame.groupby("engaged_3")["renewed"].mean().round(3).to_string())

share engaged in month 2, by the month-1 price against the middle of its band:
above the middle    0.619
at or below         0.696

mean price by that month's engagement (0 = not engaged, 1 = engaged):
     month 1  month 2  month 3
0.0    5.946    5.973    5.948
1.0    8.033    8.006    7.980

share renewed, by month-3 engagement:
engaged_3
0.0    0.598
1.0    0.809


Engaged subscribers pay about 2 more each month (8.033 against 5.946 in month 1) and renew more
often (0.809 against 0.598), so engagement is a confounder, and the analysis must adjust for it.
But a month-1 price above the band's middle also lowers the share engaged in month 2, from 0.696 to
0.619, so engagement carries part of the price's effect. One regression cannot adjust for
engagement and also let it carry that effect. These tables show associations only.

## Set up the analysis

First, describe the study (`StudyProtocol`): who is in it, when follow-up starts, what each plan
means, and what the outcome is. The description does not change the estimate. It records what the
estimate means. The comment in the cell shows the call for your own study.

In [4]:
from cleverly.datasets import pricing_protocol

# For your own study, fill in the same ten fields, in the order the output prints them:
#   target_population, eligibility: who the answer is about, and the entry rules;
#   time_zero, horizon: when follow-up starts, and when the outcome is measured;
#   treatment_strategies, treatment_versions: the pricing plans compared, and their variants;
#   outcome: the measured result;
#   intercurrent_event_handling: how events between time zero and the outcome count;
#   interference_unit: the unit whose price can affect only its own outcome;
#   assumption_rationale: why the design makes the causal assumptions believable.
# protocol = cleverly.StudyProtocol(
#     target_population=..., eligibility=(...), time_zero=..., treatment_strategies=(...),
#     treatment_versions=(...), outcome=..., horizon=..., intercurrent_event_handling=(...),
#     interference_unit=..., assumption_rationale=(...),
# )
protocol = pricing_protocol()
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 682ab02814f9837b
target population: Active monthly subscribers at the start of a three-month term
eligibility: ['Active subscription at the first monthly renewal price', 'A renewal price set by the pricing engine in each of months 1 to 3', 'One account per customer, so shared-household accounts are excluded', 'A three-month term that starts at time zero, with no exit before the month-3 renewal']
time zero: The first monthly renewal price, after the usage score and the month-1 engagement score and before the engine sets the price
treatment strategies: ["The pricing engine's own price in months 1 to 3", "Raise each monthly price by a declared factor where the raised price stays inside that month's price band, otherwise keep the engine's price. The question uses a 12% rise, and the support step also reads 6% and 25%", "Cut each monthly price by 15% where the cut price stays inside that month's price band, otherwise keep the engine's price"]
treatment versi

Follow-up starts at the first monthly price. Eligibility is a three-month term that starts at
time zero, with no exit before the month-3 renewal, so no later event decides who counts. Each
plan quotes its unrounded price for one month, and each account belongs to one customer. The ID
on the first line ties each result to this description.

Next, tell `cleverly` the price columns in time order (`treatment`), what is recorded before each
price (`baseline`, `time_varying`), and each price's band (`continuous_treatment`). Each plan is a
`DynamicRegimen` with one rule per month, fixed before the fit, and `RegimeContrast` compares each
plan with the engine's prices. `study.identify(...)` then lists what must be true for the answer
to be causal.

In [5]:
from cleverly import CausalStudy, LongitudinalTreatment, RegimeContrast
from cleverly.datasets import PRICE_BANDS, PRICE_PLANS


# For your own data, change the column names. Declare each price's band as
# cleverly.interventions.Support(lower, upper), two functions that read the history and return
# each row's limits; PRICE_BANDS holds this page's bands. Write your own plan as a DynamicRegimen
# of one ModifiedPolicy per month; the source of cleverly.datasets.price_rise shows how.
def pricing_design(outcome):
    return LongitudinalTreatment(
        outcome=outcome,
        treatment=["price_1", "price_2", "price_3"],
        baseline=["usage"],
        time_varying=[["engaged_1"], ["engaged_2"], ["engaged_3"]],
        continuous_treatment=PRICE_BANDS,
    )


study = CausalStudy(frame, design=pricing_design("renewed"), protocol=protocol)
for month in (1, 2, 3):
    print(f"history_names({month}):", study.data.history_names(month))
print()
print("plans:", list(PRICE_PLANS))
effect = study.identify(RegimeContrast(PRICE_PLANS, reference="natural_prices"))
print(effect.summary(protocol="fingerprint"))

history_names(1): ('usage', 'engaged_1')
history_names(2): ('usage', 'engaged_1', 'engaged_2')
history_names(3): ('usage', 'engaged_1', 'engaged_2', 'engaged_3')

plans: ['natural_prices', 'price_up', 'price_down']
contrast of each regime against the reference
identified by explicit-adjustment: sequential g-formula under the declared treatment regimen
history at price_1: ['usage', 'engaged_1']
history at price_2: ['usage', 'engaged_1', 'engaged_2', 'price_1']
history at price_3: ['usage', 'engaged_1', 'engaged_2', 'engaged_3', 'price_1', 'price_2']
required nuisances: ['sequential_outcome_regressions', 'treatment_and_censoring_mechanisms']
assumptions:
  - consistency: each observed history equals the potential history under its realized regimen
  - no interference: one unit's potential history does not depend on other units' regimens
  - sequential exchangeability in the form the chosen interpretation names: strong sequential randomization (Díaz et al. 2023, Assumption 3) for the modi

The `history at` lines list what the method adjusts for before each price. This page asks for the "modified treatment
policy effect": each month's engine price is replaced with the plan's price, and later engagement
and prices respond. It needs "strong sequential randomization", meaning every common cause of a
price, the later engagement and prices, and the renewal is recorded. The engine log argues for it,
and no data test exists.

"Positivity" means the plan's price falls where the engine actually prices similar subscribers.
The declared band guarantees it, and "Check the fit" shows how thin the data get. "Consistency" (a
price means the quoted price) and "no interference" (one subscriber's price does not change
another's renewal) need judgment.

:::{dropdown} Details: the second meaning of the effect
The second interpretation, a stochastic intervention (LMTP-SI), draws each month's price from the
law of the plan's price given the history. It needs only standard sequential randomization, in
which the common causes of each price and the next covariates are recorded, and it answers a
different question. Díaz, Williams, Hoffman and Schenck (2023), Theorem 1, gives both.
:::

## Effect on renewals

The method is longitudinal TMLE (LTMLE), a version of TMLE for treatments that repeat over time.
It works backward. A model of renewal given everything through the month-3 price predicts each
renewal at the plan's month-3 price. A model of that prediction given the history through month 2
predicts it at the plan's month-2 price, and so on back to month 1, so the later months respond
to each planned price. A correction step then weights each subscriber by a density ratio, how
much more likely the recorded prices are under the plan than under the engine, to remove the bias
the outcome models leave.

- `outcome_learner` fits the month-3 model and `pseudo_learner` the earlier ones.
- `treatment_learner` with `ratio="classifier"` estimates the density ratio with a classifier that
  tells the plan's prices from the engine's. Its tight tolerance makes the largest weights the same
  on every computer.
- `CrossFitting(enabled=False)`: each model trains on all rows. These models have few
  parameters, and the simulation study behind this setup fits the same way.

In [6]:
from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

# For your own data, swap in learners that suit your outcome and your prices.
logistic = LogisticRegression(max_iter=1000, random_state=42)
ratio_model = make_pipeline(
    PolynomialFeatures(2, include_bias=False),
    StandardScaler(),
    LogisticRegression(C=1e6, tol=1e-8, max_iter=100_000, random_state=42),
)
ltmle = TMLEMethod(
    models=ModelSpec(
        outcome_learner=logistic,
        pseudo_learner=LinearRegression(),
        treatment_learner=ratio_model,
        ratio="classifier",
    ),
    cross_fitting=CrossFitting(enabled=False),
    runtime=Runtime(random_state=1000, n_jobs=1),
)
result = effect.estimate(method=ltmle)
print(result.summary(protocol="fingerprint"))


def against_truth(fit, truth_kind):
    rows = []
    for key, estimate in fit.estimates.items():
        plan = key.removeprefix("ate_regimen[").removesuffix(" vs natural_prices]")
        exact = truth[f"{truth_kind}[{plan} vs natural_prices]"]
        low, high = estimate.ci
        rows.append(
            {
                "contrast": plan,
                "estimate": estimate.psi,
                "std_error": estimate.std_error,
                "ci_low": low,
                "ci_high": high,
                "truth": exact,
                "covers": bool(low <= exact <= high),
                "support": estimate.support.label,
            }
        )
    return pd.DataFrame(rows).set_index("contrast")


print()
print(against_truth(result, "ate_regimen").round(4).to_string())
print()
for key, estimate in result.estimates.items():
    print(f"{key}: {estimate.support.label}. {estimate.support.reason}")

Longitudinal TMLE (3 time points, n = 10000)

parameter                                  estimate  std. error  95% CI              p-value
-----------------------------------------  --------  ----------  ------------------  -------
ate_regimen[price_up vs natural_prices]    -0.1025   0.0065      [-0.1152, -0.0899]  <1e-4  
ate_regimen[price_down vs natural_prices]  0.1050    0.0078      [0.0896, 0.1203]    <1e-4  

  time points: 3
  outcome family: binomial
  regimens: natural_prices=(m:natural course/m:natural course/m:natural course), price_up=(m:price_up at 1/m:price_up at 2/m:price_up at 3), price_down=(m:price_down at 1/m:price_down at 2/m:price_down at 3)
    a 'd' is a rule d_t(H_t) read off [W, L_1, ..., L_t]; its followers are a covariate-dependent set, so the counts below describe this sample
    assigned arms, natural_prices: c51a991afd938ee0
    assigned arms, price_up: ed3314cb33273f4d
    assigned arms, price_down: 517e7cb67a19cdff
    a 'q' is a known policy q_t(a | H_t

In the first table, the 12% rise changes the month-3 renewal share by -0.1025 (standard error
0.0065, 95% interval -0.1152 to -0.0899), which contains the true -0.1003. The 15% cut changes it
by 0.1050 (0.0896 to 0.1203), which contains the true 0.1149. The interval is valid in large
samples when the outcome models and the density ratios are reasonably accurate. The rest of the
summary records the setup, and this page does not use its `[experimental]` bands.

`supported` in the last lines means a simulation study covers this kind of fit: this method, three
months, models trained on all rows, a 0-or-1 outcome, and classifier weights. It does not cover
these exact models or plans.

:::{dropdown} Details: the interval's condition, and repeated draws
The point estimate is consistent when every month's outcome model is right, or when every month's
density ratio is right. The Wald interval from the influence curve needs both to converge fast
enough. The earlier months' models predict a probability, and the method clips their predictions
to the range 0 to 1.

A simulation we ran for this page (script
`reviews/notebook-review/probes/sequential-pricing/sweep.py`) repeats the fits on 200 draws, seeds
1000 to 1199. The two intervals contained the truth on 186 and 187 of 200 draws. A count near the
nominal 190 has a Monte Carlo standard error of about 3 draws, so both lie about one standard
error below it. The biases were -0.0002 and 0.0004, each with a Monte Carlo standard error of
0.0005. The sweep raised `SparseInfluenceWarning` on 36 of 200 renewal fits, and this draw raised
none.
:::

## Effect on revenue

Revenue lies between 0 and 10, and the fit refuses a continuous outcome under this kind of plan
without that known range (`Targeting(q_bounds=(0.0, 10.0))`). It never clips an observed revenue.
The outcome model is a linear regression, which is cheap but cannot match revenue's shape, 0 or
the price.

In [7]:
from cleverly import Targeting

# For your own data, set q_bounds to your outcome's known range.
revenue_method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=LinearRegression(),
        pseudo_learner=LinearRegression(),
        treatment_learner=ratio_model,
        ratio="classifier",
    ),
    targeting=Targeting(q_bounds=(0.0, 10.0)),
    cross_fitting=CrossFitting(enabled=False),
    runtime=Runtime(random_state=1000, n_jobs=1),
)
revenue_study = CausalStudy(frame, design=pricing_design("revenue"), protocol=protocol)
revenue_result = revenue_study.identify(
    RegimeContrast(PRICE_PLANS, reference="natural_prices")
).estimate(method=revenue_method)
print(against_truth(revenue_result, "ate_revenue").round(4).to_string())
print()
for key, estimate in revenue_result.estimates.items():
    print(f"{key}: {estimate.support.label}. {estimate.support.reason}")

            estimate  std_error  ci_low  ci_high   truth  covers    support
contrast                                                                   
price_up     -0.3129     0.0559 -0.4224  -0.2035 -0.3122    True  supported
price_down    0.0743     0.0536 -0.0307   0.1792  0.1254    True  supported

ate_regimen[price_up vs natural_prices]: supported. Row G39.ltmle.tau3_revenue covers this configuration.
ate_regimen[price_down vs natural_prices]: supported. Row G39.ltmle.tau3_revenue covers this configuration.


The 12% rise changes mean month-3 revenue by -0.3129 per subscriber (interval -0.4224 to -0.2035),
against a true -0.3122: the subscribers who lapse cost more than the higher price brings in. The
cut's estimate, 0.0743, has an interval of -0.0307 to 0.1792, which includes zero, so this fit
cannot say whether the cut raises revenue. Its true effect is 0.1254. The study behind
`supported` does not cover a revenue contrast or a linear outcome model.

:::{dropdown} Details: the linear outcome model's bias, over repeated draws
Over the 200 draws of `sweep.py`, the estimates of the rise and the cut had biases of 0.0099 and
0.0094. Each is about two and a half Monte Carlo standard errors (0.0039 and 0.0038) and about a
sixth of the spread of the estimates (standard deviations 0.0557 and 0.0537). The intervals
contained the truth on 184 and 191 of 200 draws. The count 184 lies between one and two standard
errors below the nominal 190, and 191 lies above it. The cut's interval contained zero on 60 of
200 draws, and `SparseInfluenceWarning` came on 46 of 200 revenue fits, though not on this draw.

The probe `revenue_learner.py` refits 60 of those draws with a quadratic outcome model as well.
For the rise, the bias was 0.0028 with the quadratic model and 0.0132 with the linear one, with
Monte Carlo standard errors of 0.0075 and 0.0074. On the same draws the switch moves the rise's
estimate by 0.0104 on average (paired standard error 0.0005) and the cut's by 0.0076 (paired
standard error 0.0010). The paired standard error is the one to read, because both models fit
the same draws. So the linear model carries most of the rise's bias and about half of the
cut's. With the quadratic model, 60 draws cannot tell the rise's bias from zero.
:::

## Why one regression cannot answer a month-by-month question

The usual tool is one logistic regression of `renewed` on usage, the three engagement scores, and
the three prices: swap in the plan's prices, predict every renewal, and average the change. Here
the regression is even correctly specified, because the true model is linear on the log-odds
scale in these columns, yet it answers a different question. The code fits it
with engagement (`adjust`) and without (`omit`), and prints the exact effect with the later months
held where the engine's prices left them.

In [8]:
from cleverly.learners import thread_limit


def planned_prices(plan, month):
    price = frame[f"price_{month}"].to_numpy()
    engaged = frame[f"engaged_{month}"].to_numpy()
    if plan == "price_up":
        return np.where(price < (8 + 2 * engaged) / 1.12, 1.12 * price, price)
    return np.where(price < (4 + 2 * engaged) / 0.85, price, 0.85 * price)


def one_regression(columns, plan):
    # One CPU thread for this plain scikit-learn fit, as cleverly uses for its own fits.
    with thread_limit():
        model = LogisticRegression(max_iter=2000, C=1e6).fit(frame[columns], frame["renewed"])
        moved = frame[columns].copy()
        for month in (1, 2, 3):
            moved[f"price_{month}"] = planned_prices(plan, month)
        return float(
            np.mean(model.predict_proba(moved)[:, 1] - model.predict_proba(frame[columns])[:, 1])
        )


prices = ["price_1", "price_2", "price_3"]
adjust_columns = ["usage", "engaged_1", "engaged_2", "engaged_3", *prices]
omit_columns = ["usage", *prices]
rows = []
for plan in ("price_up", "price_down"):
    key = f"ate_regimen[{plan} vs natural_prices]"
    low, high = result[key].ci
    rows.append(
        {
            "plan": plan,
            "truth": truth[key],
            "ltmle": result[key].psi,
            "ltmle_low": low,
            "ltmle_high": high,
            "adjust": one_regression(adjust_columns, plan),
            "omit": one_regression(omit_columns, plan),
        }
    )
print(pd.DataFrame(rows).set_index("plan").round(4).to_string())
print()
print("exact effect, with the later engagement or the engine's later prices held where they were:")
paths = {
    plan: {
        "truth": truth[f"ate_regimen[{plan} vs natural_prices]"],
        "engagement held": truth[f"ate_engagement_held[{plan}]"],
        "engine held": truth[f"ate_engine_held[{plan}]"],
        "both held": truth[f"ate_both_held[{plan}]"],
    }
    for plan in ("price_up", "price_down")
}
print(pd.DataFrame(paths).round(4).to_string())

             truth   ltmle  ltmle_low  ltmle_high  adjust    omit
plan                                                             
price_up   -0.1003 -0.1025    -0.1152     -0.0899 -0.0796 -0.0202
price_down  0.1149  0.1050     0.0896      0.1203  0.0938  0.0268

exact effect, with the later engagement or the engine's later prices held where they were:
                 price_up  price_down
truth             -0.1003      0.1149
engagement held   -0.0906      0.1050
engine held       -0.0900      0.1081
both held         -0.0804      0.0973


For the 12% rise, LTMLE gives -0.1025 against the true -0.1003. The `adjust` regression gives
-0.0796, outside the LTMLE interval. A price changes the next engagement and the engine's next
price, and both change the renewal. The regression holds them where they were, so it estimates the
`both held` value, -0.0804, and not the truth. Holding only engagement gives -0.0906 and only the
engine's next price -0.0900, so each path carries about half of the gap.

The regression is correctly specified, so more data cannot close the gap. Leaving engagement out
is worse: `omit` gives -0.0202, because engaged subscribers get higher prices and renew more, so a
high price looks harmless. For the 15% cut, `adjust` gives 0.0938, inside the LTMLE interval here,
but it estimates 0.0973, further from the truth (0.1149) than LTMLE's 0.1050 is. Report the LTMLE
estimate.

:::{dropdown} Details: what the regressions estimate, and repeated draws
With engagement, the regression targets a natural-direct-effect type of quantity: the mean
renewal under the plan's prices, with the later engagement and the engine's later prices held at
their values under the engine's own prices. Hernán and Robins (*Causal Inference: What If*,
Chapter 20) call the problem treatment-confounder feedback. The regression without engagement
targets an association. Its limit, approximated from one draw of 400,000 subscribers, is -0.0203
for the rise and 0.0270 for the cut (probe `summarize.py`).

Over the 200 draws of `sweep.py`, the `adjust` regression was further from the truth than LTMLE
on 193 of 200 draws for the rise and on 185 for the cut. It lay outside the LTMLE interval on 194
draws for the rise and on 127 for the cut, and between the LTMLE and `omit` estimates on 200 and
199 of 200 draws. The `omit` regression lay outside the LTMLE interval on every draw for both
plans.
:::

## Is there enough data for a bigger price rise?

A bigger rise sends more prices past the top of the band, where the plan keeps the engine's price
(the held share). It also lifts the lowest price the plan assigns to the factor times the bottom
of the band. The exact density ratio is zero for a subscriber whose engine price lies below that
value, because the plan never produces such a price. Near the top, the plan's prices include held
and raised ones, so the exact ratio there is one plus the ratio from the raised prices, and those
subscribers carry the largest ratios. The code fits 6%, 12%, and 25% rises (`price_rise(factor)`).

In [9]:
from cleverly.datasets import price_rise

# For your own data, try the size of change the decision needs, and a smaller and a larger one.
rises = {
    "natural_prices": PRICE_PLANS["natural_prices"],
    "up_6": price_rise(1.06),
    "price_up": PRICE_PLANS["price_up"],
    "up_25": price_rise(1.25),
}
rise_fit = study.identify(RegimeContrast(rises, reference="natural_prices")).estimate(method=ltmle)
support = rise_fit.diagnostics.support().to_frame()
month3 = support[(support["time"] == 3) & (support["regimen"] != "natural_prices")]
columns = ["regimen", "time", "n_followed", "max_weight", "effective_n", "share_truncated"]
print(month3[columns].round(3).to_string(index=False))
print()
factors = {"up_6": 1.06, "price_up": 1.12, "up_25": 1.25}
ceiling = 8 + 2 * frame["engaged_3"]
lowest = 4 + 2 * frame["engaged_3"]

rows = []
for plan, factor in factors.items():
    key = f"ate_regimen[{plan} vs natural_prices]"
    low, high = rise_fit[key].ci
    rows.append(
        {
            "plan": plan,
            "zero_weight_share": float((frame["price_3"] < factor * lowest).mean()),
            "held_share": float((frame["price_3"] >= ceiling / factor).mean()),
            "estimate": rise_fit[key].psi,
            "truth": truth[key],
            "ci_width": high - low,
        }
    )
print(pd.DataFrame(rows).set_index("plan").round(4).to_string())
saturation = (
    truth["ate_regimen[up_25 vs natural_prices]"] / truth["ate_regimen[up_6 vs natural_prices]"]
)
print(f"exact effect of up_25 over up_6: {saturation:.2f}; a linear response gives {25 / 6:.2f}")
print()
for key, estimate in rise_fit.estimates.items():
    print(f"{key}: {estimate.support.label}")

 regimen  time  n_followed  max_weight  effective_n  share_truncated
    up_6     3       10000       5.712     6629.661              0.0
price_up     3       10000      13.704     3508.672              0.0
   up_25     3       10000      19.976     1977.153              0.0

          zero_weight_share  held_share  estimate   truth  ci_width
plan                                                               
up_6                 0.0559      0.0862   -0.0598 -0.0587    0.0129
price_up             0.1231      0.1916   -0.1025 -0.1003    0.0254
up_25                0.3044      0.4505   -0.1223 -0.1283    0.0393
exact effect of up_25 over up_6: 2.19; a linear response gives 4.17

ate_regimen[up_6 vs natural_prices]: supported
ate_regimen[price_up vs natural_prices]: supported
ate_regimen[up_25 vs natural_prices]: supported


The effective sample size (`effective_n`) is the number of equally weighted subscribers that would
give the same precision as the uneven weights. At month 3:

| plan | zero-weight share | held share | largest weight | effective n of 10,000 | interval width |
| --- | --- | --- | --- | --- | --- |
| `up_6` | 0.0559 | 0.0862 | 5.712 | 6630 | 0.0129 |
| `price_up` | 0.1231 | 0.1916 | 13.704 | 3509 | 0.0254 |
| `up_25` | 0.3044 | 0.4505 | 19.976 | 1977 | 0.0393 |

From 6% to 25%, the largest weight more than triples, the effective sample size falls to a fifth
of the sample, and the interval triples in width. The estimates, -0.0598, -0.1025, and -0.1223,
stay near the true -0.0587, -0.1003, and -0.1283, and each interval contains its truth. So the data
still answer for a 25% rise, but from far fewer subscribers, and that plan leaves 45% of month-3
prices unchanged. The effect flattens: the true 25% effect is 2.19 times the 6% effect, against
4.17 for a straight line. No simulation study checks the 6% or 25% plans.

## Check the fit

`result.assess()` runs the checks that apply. The overlap check (`assessment.report("support")`)
shows each plan's largest weight and effective sample size by month. Very large weights mean the
answer rests on a few subscribers.

In [10]:
assessment = result.assess()
ledger = assessment.to_frame()
ran = ledger[~ledger["status"].isin(["unavailable", "not_applicable", "deferred"])]
print(ran[["surface", "check", "status"]].to_string(index=False))
support_report = assessment.report("support").to_frame()
columns = ["regimen", "time", "n_followed", "max_weight", "effective_n", "share_truncated"]
print()
print(support_report[columns].round(3).to_string(index=False))
kish_ratio = (support_report["effective_n"] / support_report["n_followed"]).min()
print(f"minimum effective-sample-size ratio: {kish_ratio:.1%}")
held_down = (planned_prices("price_down", 3) == frame["price_3"]).mean()
print(f"price_down, share of month-3 prices left at the engine's value: {held_down:.1%}")
print()
nuisance = assessment.report("nuisance_models")
outcome = nuisance.to_frame()
outcome = outcome[(outcome["role"] == "outcome") & (outcome["regimen"] == "price_down")]
print(
    outcome[["role", "time", "n", "evaluation", "auc", "calibration_slope"]]
    .round(3)
    .to_string(index=False)
)
print(
    "not scored:",
    sorted({omission.reason for omission in nuisance.omissions if omission.role == "treatment"}),
)

   surface           check    status
validation score_equations    passed
validation         support completed
validation nuisance_models completed

       regimen  time  n_followed  max_weight  effective_n  share_truncated
natural_prices     1       10000       1.000    10000.000              0.0
natural_prices     2       10000       1.000    10000.000              0.0
natural_prices     3       10000       1.000    10000.000              0.0
      price_up     1       10000       3.352     6939.366              0.0
      price_up     2       10000       7.352     5020.246              0.0
      price_up     3       10000      13.704     3508.672              0.0
    price_down     1       10000       4.646     5207.473              0.0
    price_down     2       10000      13.093     2845.636              0.0
    price_down     3       10000      31.240     1554.276              0.0
minimum effective-sample-size ratio: 15.5%
price_down, share of month-3 prices left at the engine's v

`passed` for `score_equations` confirms that the correction step did its job, and `completed`
means a report was computed. The cut's weights compound, because each weight multiplies three
monthly ratios: its largest weight goes from 4.646 at month 1 to 13.093 at month 2 and 31.240 at
month 3. By month 3 its 10,000 subscribers count as about 1,554, the smallest share, 15.5%. It
leaves 19.7% of month-3 prices unchanged, because a 15% cut there would leave the band. No weight
was trimmed (`share_truncated` is 0).

The weights come from the classifier, which nothing here scores (the `not scored` line), so the
table cannot see an error in it. The month-3 outcome model has a calibration slope of 1.004, where
1 is ideal, and an AUC of 0.697, but it is scored on its own training rows, so this does not show
that it is right.

:::{dropdown} Details: the weights over repeated draws
In `sweep.py`, the largest month-3 weight of the rise averaged 14.82, with a standard deviation of
2.03. For the cut it averaged 31.76, with a standard deviation of 5.21. The effective sample size
over n averaged 0.361 for the rise and 0.154 for the cut.

The classifier returns a positive ratio at every price, so its weights are not the exact zeros of
the bigger-rise section. On the classifier route, `g_bounds` reaches no factor of a price node,
and these data have no censoring, so it bounds nothing here. `LTMLE(ratio_trim=q)` trims the
per-month ratios at one pooled quantile, and this page does not use it.
:::

## Try to break the result

These checks change the data in a known way and refit the 12% rise's renewal fit:

- `placebo` replaces each price with a made-up one, inside the same band, that has no effect on
  renewal, so the estimate should be about zero;
- `subset` refits on random 70% samples, so the spread should match what the standard error
  predicts for samples of that size;
- `random_common_cause` adds a column of pure noise each month, so the estimate should barely move;
- `bootstrap_stability` refits on resamples, so the spread should be about the standard error.

The cell runs 10 refits per check, half the default, to keep it near two minutes. None of these
checks can detect a missing confounder.

In [11]:
import warnings

from cleverly import SparseInfluenceWarning
from cleverly.validation import refute

estimand = "ate_regimen[price_up vs natural_prices]"
with warnings.catch_warnings():
    # A subset refit of the thin price_down tail repeats a warning about sparse weights.
    warnings.simplefilter("ignore", SparseInfluenceWarning)
    report = refute(
        result,
        estimand=estimand,
        tests=("placebo", "subset", "random_common_cause", "bootstrap_stability"),
        n_replicates=10,
        noise_nodes=True,
        random_state=1000,
    )
columns = ["test", "original", "refuted_mean", "refuted_sd", "expectation", "verdict"]
print(report.to_frame()[columns].round(4).to_string(index=False))
print()
for test in report.tests:
    if test.statistic is not None:
        pvalue = "" if test.statistic_pvalue is None else f" (p = {test.statistic_pvalue:.2f})"
        print(f"{test.name}: {test.statistic_name} = {test.statistic:.3f}{pvalue}")
    print(f"{test.name}: {len(test.values)} refits, {len(test.failures)} failed")

               test  original  refuted_mean  refuted_sd                             expectation     verdict
            placebo   -0.1025        0.0006      0.0028                                     ~ 0 descriptive
             subset   -0.1025       -0.1038      0.0042                       scatter ~ 0.00424 descriptive
random_common_cause   -0.1025       -0.1028      0.0004                               ~ -0.1025 descriptive
bootstrap_stability   -0.1025       -0.1053      0.0057 spread 0.005749 around the original fit descriptive

placebo: t = 0.720 (p = 0.49)
placebo: 10 refits, 0 failed
subset: scatter ratio rho = 0.984 (p = 0.93)
subset: 10 refits, 0 failed
random_common_cause: paired shift in standard errors = -0.034
random_common_cause: 10 refits, 0 failed
bootstrap_stability: 10 refits, 0 failed


All four behave as expected. The placebo estimates average 0.0006 (t = 0.72, p = 0.49), the 70%
refits scatter as much as sampling predicts (ratio 0.98), the noise moves the estimate by -0.034
standard errors, and the bootstrap spread is 0.88 of the standard error. The `verdict` column
reads `descriptive`: `cleverly` gives a pass or fail only when a simulation study backs the check
for this kind of fit, and none covers this one, so you judge the numbers.

:::{dropdown} Details: the placebo prices, and 400 placebo draws
At each month the placebo takes a donor's position within the donor's band and puts the same
position into the subscriber's own band. From month 2 on, the donor is one of the 5 subscribers
whose previous-month value is nearest the placebo's own previous-month value. The placebo prices
therefore still depend on engagement through the band, and a rejection could also show a
classifier that cannot follow the band.

With ten refits, the mean reported standard error of the placebo is 1.98 times the spread of its
estimates (0.0028). The probe `placebo.py` repeats the placebo with 400 draws, in about 20
minutes. There the mean is -0.00013, with t = -0.45 and p = 0.65, the mean reported standard
error is 1.01 times the spread (0.0056), and 94.5% of the 95% intervals contain 0. So the ratio
1.98 comes from the small spread of ten draws.

A simulation study calibrates this placebo for a three-month LTMLE with five-fold cross-fitting,
its own learners, and 20 draws ([roadmap](../roadmap.md), item X82, part J). This fit runs on all
rows, with other learners, at 10 draws, so it stays descriptive. The revenue fit accepts the same
four checks.
:::

## What this analysis cannot check

The key assumption fails if the engine, or a person, used an input the log does not record, such
as a competitor's price or an analyst's manual override. Such an input moves the price and the
renewal together, and no output here would show it. For a plan that changes prices over time,
`cleverly` has no bound yet on how strong a missing confounder would have to be
([roadmap](../roadmap.md), item X82, part G). "Checks this page does not run", under Go deeper,
lists the other checks that need something this design lacks.

## What we can and can't conclude

For the pricing lead: among subscribers who start a three-month term, raising each monthly price
by 12% wherever it stays inside the band would lower the month-3 renewal share by about 10
percentage points (95% interval -0.115 to -0.090), with later engagement and prices responding.
It would lower mean month-3 revenue by about 0.31 per subscriber (interval -0.42 to -0.20). A 15%
cut raises renewals by about 0.105 (0.090 to 0.120), but this analysis cannot tell whether it
raises revenue.

The intervals cover sampling variation only. They assume, without a check from the data, that the
engine log records every common cause of the price and the renewal, that a price means the quoted
price, and that one subscriber's price does not change another's renewal. Each plan leaves about a
fifth of the prices at the engine's value, so this is not a price elasticity, an optimal price, or
the effect of a price outside the band. One simulated sample with a known answer does not show how
the method performs in general.

## Go deeper

### How we know the method works

A simulation study, [MTP pricing](../technical-reference/method-evidence/mtp-pricing.md), checks
this method on data simulated like this page's, against an independent R implementation. It
measures both plans at three months with the exact density ratio and models like this page's. It
leaves four of this page's choices unchecked. Its learned density ratio is measured only for the
mean renewal under the rise, not for a contrast or for the cut, and it comes from a screened
quadratic classifier with indicators for the pieces of the plan, which this page's classifier
lacks. One cell measures the mean revenue under the rise with a correct outcome model, and none
measures a revenue contrast, the cut's revenue, a linear outcome model, or the 6% and 25% rises.

The repeated-draw numbers in the Details blocks come from scripts under
`reviews/notebook-review/probes/sequential-pricing/`, which repeat this page's fits on 200 draws.
They are not registered studies. `help(make_sequential_pricing)` gives the simulation's exact
formulas. The [technical entry](../technical-reference/longitudinal-tmle.md) lists the method's
evidence and limits, including the limits for a revenue outcome and a learned ratio.

### Other estimators for this question

Three other estimators could answer the same question, and the
[technical entry](../technical-reference/longitudinal-tmle.md) gives their conditions.

| estimator | what it uses | when to use it |
| --- | --- | --- |
| LTMLE | the outcome models and the density ratios, with a correction step | this page's choice. It stays on target when every outcome model is right, or when every density ratio is right |
| SDR | the same, with no correction step | when the outcome model or the ratio could be wrong at different months, because it needs only one of the two at each month. Its interval runs short under heavy weights, which the technical entry measures on other data |
| g-computation | the outcome models only | when every outcome model is right. It gives no protection against a wrong model |
| IPW | the density ratios only | not for this plan: `IPWMethod` refuses it, because `SequentialIPW` estimates no density ratio for a rule like this one |

A fixed price for every subscriber would answer a different question, and the fit refuses one on a
continuous price, because no regular estimator of its mean exists.

### Checks this page does not run

Dropping a recorded variable to see how far the estimate moves does not work here. The fit
refuses an empty baseline, so `usage` cannot go ([roadmap](../roadmap.md), item X63, part (c)),
and each month's engagement defines the price band. A negative-control outcome needs a measure
taken before month 1 that the prices cannot affect and the fit does not adjust for, and `usage` is
adjusted for. `future_treatment` needs a 0-or-1 treatment at the later month, and the prices are
continuous.

The generated-outcome checks (`dummy_outcome`, `simulated_outcome`, `plasmode`) draw a Gaussian
outcome, which does not fit the 0-or-1 renewal, and for revenue it draws outside the declared range
of 0 to 10, so `refute` refuses them. The E-value has no derivation for a month-by-month target
(roadmap F16). `simulated_confounding` supports a hidden variable only when no covariate is
recorded after the first price, and `engaged_2` and `engaged_3` are (roadmap F13).

## Where to go next

The [sequential offers example](sequential-offers.ipynb) applies rules that choose among offers in
the same retailer program, and it adds subscribers who leave early and a paired comparison of two
rules. The [time-to-event example](longitudinal-survival.ipynb) handles disenrollment and a
competing event. The [longitudinal user guide](../user-guide/longitudinal.md) shows how to read a
panel and how to report the simpler comparisons. The
[technical entry](../technical-reference/longitudinal-tmle.md) states the estimator for a price
plan, in its section on modified treatment policies at a node. The
[examples index](index.md) lists every example, and its retailer-program section says how this
page's simulation differs from the offers example's.